<a href="https://colab.research.google.com/github/gocleanwater/AI-class/blob/main/week7/wine_DL_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시험 끝나고 해당 코드 다시 복습 및 정리하기...

In [20]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
from torchsummary import summary
import numpy as np

In [21]:
path = '/content/drive/MyDrive/Colab Notebooks/AI-class/week3/wine.csv'
df = pd.read_csv(path)
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


In [22]:
df.columns

Index(['Wine', 'Alcohol', 'Malic.acid', 'Ash', 'Acl', 'Mg', 'Phenols',
       'Flavanoids', 'Nonflavanoid.phenols', 'Proanth', 'Color.int', 'Hue',
       'OD', 'Proline'],
      dtype='object')

In [23]:
# 결측치 확인
df.isnull().sum()

,0
Wine,0
Alcohol,0
Malic.acid,0
Ash,0
Acl,0
Mg,0
Phenols,0
Flavanoids,0
Nonflavanoid.phenols,0
Proanth,0


In [24]:
# 데이터와 타겟 분리
X = df.drop('Wine', axis=1).values
y = df['Wine'].values

In [25]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [26]:
# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [27]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape,

((142, 13), (36, 13), (142,), (36,))

In [28]:
# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.int64)

In [29]:
# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [30]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([142, 13]),
 torch.Size([36, 13]),
 torch.Size([142]),
 torch.Size([36]))

# 모델 정의

In [31]:
class WineClassificationDense(nn.Module):
    def __init__(self):
        super(WineClassificationDense, self).__init__()
        self.fc1 = nn.Linear(13, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 3)  # 4 classes in the dataset

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize the model, loss function, and optimizer
model = WineClassificationDense()

# 손실 함수 및 최적화 기법 정의

In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모델 학습

In [33]:
# Variables to store loss and accuracy
train_losses = []
test_accuracies = []

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Calculate average loss over an epoch
    train_losses.append(running_loss / len(train_dataloader))

    # Evaluate on test data
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    test_accuracies.append(accuracy)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")

print("Training complete.")

Epoch 1/20, Loss: 1.0543, Accuracy: 88.89%
Epoch 2/20, Loss: 0.9873, Accuracy: 94.44%
Epoch 3/20, Loss: 0.9216, Accuracy: 94.44%
Epoch 4/20, Loss: 0.8455, Accuracy: 97.22%
Epoch 5/20, Loss: 0.7604, Accuracy: 97.22%
Epoch 6/20, Loss: 0.6643, Accuracy: 97.22%
Epoch 7/20, Loss: 0.5930, Accuracy: 97.22%
Epoch 8/20, Loss: 0.5115, Accuracy: 97.22%
Epoch 9/20, Loss: 0.4278, Accuracy: 97.22%
Epoch 10/20, Loss: 0.3717, Accuracy: 97.22%
Epoch 11/20, Loss: 0.2955, Accuracy: 97.22%
Epoch 12/20, Loss: 0.2421, Accuracy: 97.22%
Epoch 13/20, Loss: 0.2056, Accuracy: 97.22%
Epoch 14/20, Loss: 0.1766, Accuracy: 97.22%
Epoch 15/20, Loss: 0.1431, Accuracy: 100.00%
Epoch 16/20, Loss: 0.1158, Accuracy: 100.00%
Epoch 17/20, Loss: 0.1059, Accuracy: 100.00%
Epoch 18/20, Loss: 0.0921, Accuracy: 100.00%
Epoch 19/20, Loss: 0.0810, Accuracy: 100.00%
Epoch 20/20, Loss: 0.0677, Accuracy: 100.00%
Training complete.


# 모델 평가

In [34]:
# # Evaluation
# model.eval()
# all_labels = []
# all_predictions = []
# with torch.no_grad():
#     for inputs, labels in test_dataloader:
#         outputs = model(inputs)
#         _, predicted = torch.max(outputs.data, 1)
#         all_labels.extend(labels.cpu().numpy())
#         all_predictions.extend(predicted.cpu().numpy())

# # Convert to numpy arrays
# all_labels = np.array(all_labels)
# all_predictions = np.array(all_predictions)

# # Calculate metrics
# conf_matrix = confusion_matrix(all_labels, all_predictions)
# f1 = f1_score(all_labels, all_predictions, average='weighted')
# precision = precision_score(all_labels, all_predictions, average='weighted')
# recall = recall_score(all_labels, all_predictions, average='weighted')

# # Calculate specificity for each class
# specificity = []
# for i in range(conf_matrix.shape[0]):
#     tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
#     fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
#     specificity.append(tn / (tn + fp))

# print(f'Confusion Matrix:\n{conf_matrix}')
# print(f'F1 Score: {f1:.2f}')
# print(f'Precision: {precision:.2f}')
# print(f'Recall: {recall:.2f}')
# print(f'Specificity: {np.mean(specificity):.2f}')

In [35]:
# # Plot the loss and accuracy
# plt.figure(figsize=(12, 5))

# # Plot loss
# plt.subplot(1, 2, 1)
# plt.plot(train_losses, label='Training Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Over Epochs')
# plt.legend()

# # Plot accuracy
# plt.subplot(1, 2, 2)
# plt.plot(test_accuracies, label='Test Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy (%)')
# plt.title('Test Accuracy Over Epochs')
# plt.legend()

# plt.show()

In [36]:
# # 데이터와 타겟 분리
# X = df.drop('Wine', axis=1).values
# y = df['Wine'].values

In [37]:
# # Standardize the data
# scaler = StandardScaler()
# X = scaler.fit_transform(X)

In [38]:
# data_array = np.hstack((X, y.reshape(-1, 1)))

In [39]:
# data_array.shape

In [40]:
# # Split sequences function
# def split_sequences(sequences, n_steps):
#     X, y = list(), list()
#     for i in range(len(sequences)):
#         end_ix = i + n_steps
#         if end_ix > len(sequences):
#             break
#         seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix-1, -1]
#         X.append(seq_x)
#         y.append(seq_y)
#     return np.array(X), np.array(y)

# # Apply sequence transformation
# n_steps = 5
# X, y = split_sequences(data_array, n_steps)

# # Split the dataset into training and test sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [41]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

(torch.Size([142, 13]),
 torch.Size([142]),
 torch.Size([36, 13]),
 torch.Size([36]))

In [42]:
# # Convert to PyTorch tensors
# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.int64)
# X_test = torch.tensor(X_test, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.int64)

# # Create DataLoader
# train_dataset = TensorDataset(X_train, y_train)
# train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# test_dataset = TensorDataset(X_test, y_test)
# test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [43]:
# # Define the 1D CNN model
# class WineClassificationCNN(nn.Module):
#     def __init__(self):
#         super(WineClassificatonCNN, self).__init__()
#         self.conv1 = nn.Conv1d(13, 16, kernel_size=3, padding=1)  # Change input channels to 13
#         self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
#         self.fc1 = nn.Linear(32 * 5, 64)  # Adjust the linear layer input size accordingly
#         self.fc2 = nn.Linear(64, 3)  # 3 classes in the dataset

#     def forward(self, x):
#         x = torch.relu(self.conv1(x))
#         x = torch.relu(self.conv2(x))
#         x = x.view(x.size(0), -1)
#         x = torch.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

In [44]:
# # Initialize the model, loss function, and optimizer
# model = WineClassificatonCNN()

In [45]:
# # Print the summary of the model
# summary(model, input_size=(6, 5))

In [46]:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # Variables to store loss and accuracy
# train_losses = []
# test_accuracies = []

In [47]:
# # Training loop
# num_epochs = 20
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     for inputs, labels in train_dataloader:
#         # Zero the parameter gradients
#         optimizer.zero_grad()

#         # Forward pass
#         inputs = inputs.permute(0, 2, 1)  # Change shape to (batch_size, channels, sequence_length)
#         outputs = model(inputs)
#         loss = criterion(outputs, labels)

#         # Backward pass and optimize
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#     # Calculate average loss over an epoch
#     train_losses.append(running_loss / len(train_dataloader))

#     # Evaluate on test data
#     model.eval()
#     correct = 0
#     total = 0
#     all_labels = []
#     all_predictions = []
#     with torch.no_grad():
#         for inputs, labels in test_dataloader:
#             inputs = inputs.permute(0, 2, 1)  # Change shape to (batch_size, channels, sequence_length)
#             outputs = model(inputs)
#             _, predicted = torch.max(outputs.data, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()
#             all_labels.extend(labels.cpu().numpy())
#             all_predictions.extend(predicted.cpu().numpy())

#     accuracy = 100 * correct / total
#     test_accuracies.append(accuracy)

#     print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")

# print("Training complete.")

# # Convert lists to numpy arrays
# all_labels = np.array(all_labels)
# all_predictions = np.array(all_predictions)

# # Calculate metrics
# conf_matrix = confusion_matrix(all_labels, all_predictions)
# f1 = f1_score(all_labels, all_predictions, average='weighted')
# precision = precision_score(all_labels, all_predictions, average='weighted')
# recall = recall_score(all_labels, all_predictions, average='weighted')

# # Calculate specificity for each class
# specificity = []
# for i in range(conf_matrix.shape[0]):
#     tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
#     fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
#     specificity.append(tn / (tn + fp))

# # Print metrics
# print(f'Confusion Matrix:\n{conf_matrix}')
# print(f'F1 Score: {f1:.2f}')
# print(f'Precision: {precision:.2f}')
# print(f'Recall: {recall:.2f}')
# print(f'Specificity: {np.mean(specificity):.2f}')

# # Plot the loss and accuracy
# plt.figure(figsize=(12, 5))

# # Plot loss
# plt.subplot(1, 2, 1)
# plt.plot(train_losses, label='Training Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Over Epochs')
# plt.legend()

# # Plot accuracy
# plt.subplot(1, 2, 2)
# plt.plot(test_accuracies, label='Test Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy (%)')
# plt.title('Test Accuracy Over Epochs')
# plt.legend()

# plt.show()